# Phase 3: Error Classifier Training\n\nTrain a dual-image + text multi-label classifier using BLIP-2 vision encoder and XLM-RoBERTa.\n\n**Input:** `phase2_annotated.parquet` + images\n**Output:** `best_model/` + `training_metrics.json` + `splits/`

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes scikit-multilearn scikit-learn torch torchvision pandas matplotlib

In [ ]:
# Cell 2: Config & Imports
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, roc_auc_score

# ── Path Detection ──
IS_KAGGLE = os.path.exists("/kaggle/working")
OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "./output"
INPUT_DIR = "/kaggle/input" if IS_KAGGLE else "./output"

PHASE2_DIR = os.path.join(INPUT_DIR, "phase2-output") if IS_KAGGLE else OUTPUT_DIR
# Phase 1 images
PHASE1_DIR = os.path.join(INPUT_DIR, "phase1-output") if IS_KAGGLE else OUTPUT_DIR

# ── Constants ──
ERROR_TAXONOMY = {
    0: "Wrong Object", 1: "Missing Object", 2: "Extra Object",
    3: "Wrong Attribute", 4: "Spatial Error", 5: "Style Mismatch",
    6: "Over-editing", 7: "Under-editing", 8: "Artifact/Quality",
    9: "Ambiguous Prompt", 10: "Failed Removal",
}
NUM_ERROR_CLASSES = 11

BLIP2_MODEL = "Salesforce/blip2-opt-2.7b"
XLM_ROBERTA_MODEL = "xlm-roberta-base"
IMAGE_SIZE = 224
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 8
EPOCHS = 20
EARLY_STOP_PATIENCE = 3
LR_HEAD = 1e-4
LR_ENCODER = 1e-5
RANDOM_SEED = 42

# Directories
MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
SPLITS_DIR = os.path.join(OUTPUT_DIR, "splits")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PHASE2_DIR: {PHASE2_DIR}")
print(f"PHASE1_DIR: {PHASE1_DIR}")

In [ ]:
# Cell 3: Load Phase 2 data and create stratified splits
from skmultilearn.model_selection import iterative_train_test_split
from scipy.sparse import lil_matrix

# Load data
parquet_path = os.path.join(PHASE2_DIR, "phase2_annotated.parquet")
df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df)} annotated samples")

# Parse error labels
df["error_labels_list"] = df["error_labels"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# Filter out invalid rows
valid_mask = df["error_labels_list"].apply(lambda x: x is not None and len(x) == NUM_ERROR_CLASSES)
df = df[valid_mask].reset_index(drop=True)
print(f"Valid samples after filtering: {len(df)}")

# Build label matrix
labels = np.array(df["error_labels_list"].tolist())
indices = np.arange(len(df)).reshape(-1, 1)

# Stratified split: 80/10/10
X_train_val, y_train_val, X_test, y_test = iterative_train_test_split(
    indices, lil_matrix(labels), test_size=0.1
)
X_train, y_train, X_val, y_val = iterative_train_test_split(
    X_train_val, y_train_val, test_size=0.111  # 0.111 of 0.9 ≈ 0.1 of total
)

train_idx = X_train.flatten().tolist()
val_idx = X_val.flatten().tolist()
test_idx = X_test.flatten().tolist()

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Save splits
splits = {"train": train_idx, "val": val_idx, "test": test_idx}
with open(os.path.join(SPLITS_DIR, "splits.json"), "w") as f:
    json.dump(splits, f)
print(f"Splits saved to {SPLITS_DIR}/splits.json")

In [ ]:
# Cell 4: Dataset class
from torchvision import transforms

class EditErrorDataset(Dataset):
    """Dataset for image editing error classification."""
    
    def __init__(self, dataframe, indices, image_base_dir, split="train", languages=None):
        self.df = dataframe.iloc[indices].reset_index(drop=True)
        self.image_base_dir = image_base_dir
        self.split = split
        self.languages = languages or ["en", "hi", "bn", "ne"]
        
        # Image transforms
        if split == "train":
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.RandomHorizontalFlip(p=0.3),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225]),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225]),
            ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load images
        orig_path = os.path.join(self.image_base_dir, row["original_image_path"])
        edit_path = os.path.join(self.image_base_dir, row["edited_image_path"])
        
        orig_img = Image.open(orig_path).convert("RGB")
        edit_img = Image.open(edit_path).convert("RGB")
        
        orig_tensor = self.transform(orig_img)
        edit_tensor = self.transform(edit_img)
        
        # Select prompt language (random during training, English for eval)
        if self.split == "train":
            lang = random.choice(self.languages)
        else:
            lang = "en"
        
        if lang == "en":
            prompt = row["edit_prompt"]
        else:
            prompt = row.get(f"edit_prompt_{lang}", row["edit_prompt"])
        
        # Labels
        labels = row["error_labels_list"]
        if isinstance(labels, str):
            labels = json.loads(labels)
        labels = torch.tensor(labels, dtype=torch.float32)
        
        return {
            "original_image": orig_tensor,
            "edited_image": edit_tensor,
            "edit_prompt": prompt,
            "labels": labels,
            "sample_id": row.get("sample_id", str(idx)),
        }

# Create datasets
train_dataset = EditErrorDataset(df, train_idx, PHASE1_DIR, split="train")
val_dataset = EditErrorDataset(df, val_idx, PHASE1_DIR, split="val")
test_dataset = EditErrorDataset(df, test_idx, PHASE1_DIR, split="test")

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Test loading one sample
sample = train_dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"Image shape: {sample['original_image'].shape}")
print(f"Labels shape: {sample['labels'].shape}")
print(f"Prompt: {sample['edit_prompt'][:80]}...")

In [ ]:
# Cell 5: Model Architecture
from transformers import Blip2Model, Blip2Processor, XLMRobertaModel, XLMRobertaTokenizer


class DualImageAttention(nn.Module):
    """Cross-attention between edited (query) and original (key/value) features."""
    
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, edited_feat, original_feat):
        # edited_feat, original_feat: (batch, dim)
        # Reshape to (batch, 1, dim) for attention
        q = edited_feat.unsqueeze(1)
        kv = original_feat.unsqueeze(1)
        attn_out, _ = self.attn(q, kv, kv)
        out = self.norm(attn_out.squeeze(1) + edited_feat)
        return out


class DualImageTextClassifier(nn.Module):
    """Multi-label classifier using BLIP-2 vision + XLM-RoBERTa text encoders."""
    
    def __init__(self, num_classes=NUM_ERROR_CLASSES, vision_dim=1408, text_dim=768):
        super().__init__()
        
        # Vision encoder (BLIP-2 ViT) - will be loaded separately and frozen
        self.vision_dim = vision_dim
        self.text_dim = text_dim
        
        # Vision projection
        self.vision_proj = nn.Linear(vision_dim, 512)
        
        # Text projection  
        self.text_proj = nn.Linear(text_dim, 512)
        
        # Cross-attention for image delta
        self.delta_attention = DualImageAttention(512, num_heads=8)
        
        # Fusion: [orig_feat, edit_feat, delta_feat, text_feat] -> classifier
        self.classifier = nn.Sequential(
            nn.Linear(512 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    
    def forward(self, orig_vision_feat, edit_vision_feat, text_feat):
        """
        orig_vision_feat: (batch, vision_dim) from BLIP-2
        edit_vision_feat: (batch, vision_dim) from BLIP-2
        text_feat: (batch, text_dim) from XLM-RoBERTa
        """
        # Project
        orig_proj = self.vision_proj(orig_vision_feat)
        edit_proj = self.vision_proj(edit_vision_feat)
        text_proj = self.text_proj(text_feat)
        
        # Cross-attention delta
        delta = self.delta_attention(edit_proj, orig_proj)
        
        # Fusion
        fused = torch.cat([orig_proj, edit_proj, delta, text_proj], dim=-1)
        logits = self.classifier(fused)
        return logits


print("Model architecture defined.")
print(f"Classifier: vision_dim=1408 -> 512, text_dim=768 -> 512, fusion=2048 -> 512 -> {NUM_ERROR_CLASSES}")

In [ ]:
# Cell 6: Load encoders
print("Loading BLIP-2 vision encoder...")
blip2_processor = Blip2Processor.from_pretrained(BLIP2_MODEL)
blip2_model = Blip2Model.from_pretrained(BLIP2_MODEL, torch_dtype=torch.float16)
# Extract just the vision model
blip2_vision = blip2_model.vision_model.to(device).half()
blip2_vision.eval()
for param in blip2_vision.parameters():
    param.requires_grad = False
# Clean up LLM part to save memory
del blip2_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("Loading XLM-RoBERTa...")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained(XLM_ROBERTA_MODEL)
xlm_model = XLMRobertaModel.from_pretrained(XLM_ROBERTA_MODEL).to(device)
xlm_model.eval()

# Freeze all but last 2 layers
for name, param in xlm_model.named_parameters():
    param.requires_grad = False
for name, param in xlm_model.encoder.layer[-2:].named_parameters():
    param.requires_grad = True

trainable_xlm = sum(p.numel() for p in xlm_model.parameters() if p.requires_grad)
print(f"XLM-RoBERTa trainable params: {trainable_xlm:,}")

if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 7: Feature extraction helpers & training setup

@torch.no_grad()
def extract_vision_features(images_tensor):
    """Extract BLIP-2 vision features from image tensor batch."""
    # images_tensor: (batch, 3, 224, 224) - already normalized
    # BLIP-2 ViT expects pixel values, need to re-process through BLIP processor
    # Use the vision model directly with our tensors
    pixel_values = images_tensor.to(device).half()
    outputs = blip2_vision(pixel_values=pixel_values)
    # Use CLS token or pooled output
    pooled = outputs.pooler_output  # (batch, 1408)
    return pooled.float()


def extract_text_features(prompts):
    """Extract XLM-RoBERTa features from text prompts."""
    inputs = xlm_tokenizer(
        prompts, return_tensors="pt", padding=True, 
        truncation=True, max_length=128
    ).to(device)
    outputs = xlm_model(**inputs)
    # CLS token
    cls_feat = outputs.last_hidden_state[:, 0, :]  # (batch, 768)
    return cls_feat


def collate_fn(batch):
    """Custom collate that keeps prompts as strings."""
    return {
        "original_image": torch.stack([b["original_image"] for b in batch]),
        "edited_image": torch.stack([b["edited_image"] for b in batch]),
        "edit_prompt": [b["edit_prompt"] for b in batch],
        "labels": torch.stack([b["labels"] for b in batch]),
        "sample_id": [b["sample_id"] for b in batch],
    }


# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

# Initialize classifier
classifier = DualImageTextClassifier().to(device)
trainable_cls = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f"Classifier trainable params: {trainable_cls:,}")

# Class weights for imbalanced labels
all_labels = np.array(df.iloc[train_idx]["error_labels_list"].tolist())
pos_counts = all_labels.sum(axis=0)
neg_counts = len(all_labels) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)
pos_weight = torch.clamp(pos_weight, min=1.0, max=20.0)
print(f"Pos weights: {pos_weight.cpu().numpy().round(2)}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer with parameter groups
optimizer = torch.optim.AdamW([
    {"params": classifier.parameters(), "lr": LR_HEAD},
    {"params": [p for p in xlm_model.parameters() if p.requires_grad], "lr": LR_ENCODER},
], weight_decay=0.01)

# Scheduler
total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
warmup_steps = int(0.1 * total_steps)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR_HEAD, LR_ENCODER], total_steps=total_steps,
    pct_start=warmup_steps / total_steps, anneal_strategy="cos"
)

# Mixed precision scaler
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print(f"Total training steps: {total_steps}, Warmup: {warmup_steps}")
print("Training setup complete.")

In [ ]:
# Cell 8: Training loop

def evaluate(loader):
    """Run evaluation on a data loader."""
    classifier.eval()
    xlm_model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0
    
    with torch.no_grad():
        for batch in loader:
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                orig_feat = extract_vision_features(batch["original_image"])
                edit_feat = extract_vision_features(batch["edited_image"])
                text_feat = extract_text_features(batch["edit_prompt"])
                
                labels = batch["labels"].to(device)
                logits = classifier(orig_feat, edit_feat, text_feat)
                loss = criterion(logits, labels)
            
            total_loss += loss.item() * labels.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    avg_loss = total_loss / len(loader.dataset)
    
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    h_loss = hamming_loss(all_labels, all_preds)
    exact = accuracy_score(all_labels, all_preds)
    
    return {
        "loss": avg_loss,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1,
        "hamming_loss": h_loss,
        "exact_match": exact,
        "y_true": all_labels,
        "y_pred": all_preds,
        "y_prob": all_probs,
    }


# Training
history = {"train_loss": [], "val_loss": [], "val_weighted_f1": [], "val_macro_f1": []}
best_val_f1 = 0
patience_counter = 0
global_step = 0

print(f"Starting training for {EPOCHS} epochs...")
print(f"Batch size: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM_STEPS}, Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

for epoch in range(EPOCHS):
    classifier.train()
    xlm_model.train()
    epoch_loss = 0
    optimizer.zero_grad()
    
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in pbar:
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            orig_feat = extract_vision_features(batch["original_image"])
            edit_feat = extract_vision_features(batch["edited_image"])
            text_feat = extract_text_features(batch["edit_prompt"])
            
            labels = batch["labels"].to(device)
            logits = classifier(orig_feat, edit_feat, text_feat)
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS
        
        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACCUM_STEPS
        
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(classifier.parameters()) + [p for p in xlm_model.parameters() if p.requires_grad],
                max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1
        
        pbar.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM_STEPS:.4f}"})
    
    avg_train_loss = epoch_loss / len(train_loader)
    
    # Validation
    val_metrics = evaluate(val_loader)
    
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_weighted_f1"].append(val_metrics["weighted_f1"])
    history["val_macro_f1"].append(val_metrics["macro_f1"])
    
    print(f"\nEpoch {epoch+1}: train_loss={avg_train_loss:.4f}, "
          f"val_loss={val_metrics['loss']:.4f}, "
          f"val_wF1={val_metrics['weighted_f1']:.4f}, "
          f"val_mF1={val_metrics['macro_f1']:.4f}, "
          f"hamming={val_metrics['hamming_loss']:.4f}")
    
    # Early stopping / best model
    if val_metrics["weighted_f1"] > best_val_f1:
        best_val_f1 = val_metrics["weighted_f1"]
        patience_counter = 0
        torch.save({
            "classifier": classifier.state_dict(),
            "xlm_model": {k: v for k, v in xlm_model.state_dict().items()},
            "epoch": epoch,
            "best_val_f1": best_val_f1,
        }, os.path.join(MODEL_DIR, "best_checkpoint.pt"))
        print(f"  -> New best model saved (wF1={best_val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  -> Early stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete. Best val weighted F1: {best_val_f1:.4f}")

In [ ]:
# Cell 9: Test evaluation with best model
# Load best checkpoint
checkpoint = torch.load(os.path.join(MODEL_DIR, "best_checkpoint.pt"), map_location=device)
classifier.load_state_dict(checkpoint["classifier"])
xlm_model.load_state_dict(checkpoint["xlm_model"])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Evaluate on test set
test_metrics = evaluate(test_loader)

# Compute full metrics
from sklearn.metrics import precision_score, recall_score

per_class_f1 = f1_score(test_metrics["y_true"], test_metrics["y_pred"], average=None, zero_division=0)

full_metrics = {
    "weighted_f1": test_metrics["weighted_f1"],
    "macro_f1": test_metrics["macro_f1"],
    "micro_f1": f1_score(test_metrics["y_true"], test_metrics["y_pred"], average="micro", zero_division=0),
    "hamming_loss": test_metrics["hamming_loss"],
    "exact_match": test_metrics["exact_match"],
    "precision_macro": precision_score(test_metrics["y_true"], test_metrics["y_pred"], average="macro", zero_division=0),
    "recall_macro": recall_score(test_metrics["y_true"], test_metrics["y_pred"], average="macro", zero_division=0),
    "per_class_f1": {ERROR_TAXONOMY[i]: float(f) for i, f in enumerate(per_class_f1)},
}

# AUC-ROC
try:
    full_metrics["auc_roc_macro"] = roc_auc_score(
        test_metrics["y_true"], test_metrics["y_prob"], average="macro"
    )
except ValueError:
    full_metrics["auc_roc_macro"] = None

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
for k, v in full_metrics.items():
    if k != "per_class_f1":
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nPer-class F1:")
for name, f1 in full_metrics["per_class_f1"].items():
    print(f"  {name}: {f1:.4f}")

In [ ]:
# Cell 10: Save model, config, and metrics
# Save model config
model_config = {
    "num_classes": NUM_ERROR_CLASSES,
    "vision_dim": 1408,
    "text_dim": 768,
    "blip2_model": BLIP2_MODEL,
    "xlm_model": XLM_ROBERTA_MODEL,
    "image_size": IMAGE_SIZE,
    "error_taxonomy": ERROR_TAXONOMY,
}

with open(os.path.join(MODEL_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

# Save training metrics
training_output = {
    "history": history,
    "test_metrics": {k: v for k, v in full_metrics.items() 
                     if not isinstance(v, np.ndarray)},
    "best_epoch": int(checkpoint["epoch"]),
    "best_val_f1": float(best_val_f1),
}

with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
    json.dump(training_output, f, indent=2)

print(f"Model saved to {MODEL_DIR}/")
print(f"Metrics saved to {OUTPUT_DIR}/training_metrics.json")

In [ ]:
# Cell 11: Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training/Validation Loss
epochs_range = range(1, len(history["train_loss"]) + 1)
axes[0, 0].plot(epochs_range, history["train_loss"], "b-o", label="Train Loss")
axes[0, 0].plot(epochs_range, history["val_loss"], "r-o", label="Val Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Validation F1 Scores
axes[0, 1].plot(epochs_range, history["val_weighted_f1"], "g-o", label="Weighted F1")
axes[0, 1].plot(epochs_range, history["val_macro_f1"], "m-o", label="Macro F1")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("F1 Score")
axes[0, 1].set_title("Validation F1 Scores")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Per-class F1 Bar Chart
class_names = list(full_metrics["per_class_f1"].keys())
class_f1s = list(full_metrics["per_class_f1"].values())
colors = plt.cm.Set3(np.linspace(0, 1, len(class_names)))
axes[1, 0].barh(class_names, class_f1s, color=colors)
axes[1, 0].set_xlabel("F1 Score")
axes[1, 0].set_title("Per-Class F1 (Test Set)")
axes[1, 0].set_xlim(0, 1)
axes[1, 0].invert_yaxis()

# Confusion-style heatmap (label co-occurrence in predictions)
pred_cooccurrence = test_metrics["y_pred"].T @ test_metrics["y_pred"]
np.fill_diagonal(pred_cooccurrence, 0)
im = axes[1, 1].imshow(pred_cooccurrence, cmap="YlOrRd", aspect="auto")
axes[1, 1].set_xticks(range(NUM_ERROR_CLASSES))
axes[1, 1].set_yticks(range(NUM_ERROR_CLASSES))
axes[1, 1].set_xticklabels([ERROR_TAXONOMY[i][:8] for i in range(NUM_ERROR_CLASSES)], rotation=45, ha="right", fontsize=7)
axes[1, 1].set_yticklabels([ERROR_TAXONOMY[i][:8] for i in range(NUM_ERROR_CLASSES)], fontsize=7)
axes[1, 1].set_title("Predicted Error Co-occurrence")
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_visualization.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved.")